# Lesson 7: Model Saving and Loading

**Duration:** 15 minutes (plus wrap-up)

## Learning Objectives
- Save trained models for later use
- Load saved models for inference
- Understand different checkpoint strategies
- Document model metadata for reproducibility

## 7.1 Saving Models

Two main approaches:
1. **Save state dict only** (weights and biases) - smaller file, faster
2. **Save entire model** (architecture + weights) - more portable but larger

In [ ]:
["import torch\nimport torch.nn as nn\nimport torchvision.models as models\n\n# Create a simple model\nmodel = models.resnet18(pretrained=False)\nmodel.fc = nn.Linear(model.fc.in_features, 10)\n\n# Method 1: Save only state dict (RECOMMENDED)\ntorch.save(model.state_dict(), 'model_weights.pth')\nprint('Model weights saved to model_weights.pth')\nprint(f'File size: {open(\"model_weights.pth\").read()} bytes')\n\n# Method 2: Save entire model\ntorch.save(model, 'model_full.pth')\nprint('\\nFull model saved to model_full.pth')\n\n# Method 3: Save checkpoint with metadata\ncheckpoint = {\n    'epoch': 10,\n    'model_state_dict': model.state_dict(),\n    'optimizer_state_dict': None,  # Would save optimizer state here\n    'loss': 0.5,\n    'accuracy': 0.95,\n    'model_architecture': 'resnet18',\n}\ntorch.save(checkpoint, 'checkpoint.pth')\nprint('\\nCheckpoint with metadata saved to checkpoint.pth')"]]

## 7.2 Loading Models

Reverse the saving process:

In [ ]:
["# Method 1: Load state dict\nmodel_new = models.resnet18(pretrained=False)\nmodel_new.fc = nn.Linear(model_new.fc.in_features, 10)\nmodel_new.load_state_dict(torch.load('model_weights.pth'))\nmodel_new.eval()  # Set to evaluation mode\nprint('Model weights loaded from model_weights.pth')\n\n# Method 2: Load entire model\nmodel_loaded = torch.load('model_full.pth')\nmodel_loaded.eval()\nprint('\\nFull model loaded from model_full.pth')\n\n# Method 3: Load checkpoint with metadata\ncheckpoint = torch.load('checkpoint.pth')\nmodel_from_checkpoint = models.resnet18(pretrained=False)\nmodel_from_checkpoint.fc = nn.Linear(model_from_checkpoint.fc.in_features, 10)\nmodel_from_checkpoint.load_state_dict(checkpoint['model_state_dict'])\nmodel_from_checkpoint.eval()\n\nprint('\\nCheckpoint loaded!')\nprint(f'  Epoch: {checkpoint[\"epoch\"]}')\nprint(f'  Accuracy: {checkpoint[\"accuracy\"]:.4f}')\nprint(f'  Architecture: {checkpoint[\"model_architecture']}')"]]

## 7.3 Inference with Loaded Model

Use your loaded model to make predictions:

In [ ]:
["# Create dummy input\ninput_tensor = torch.randn(1, 3, 224, 224)\n\n# Inference\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nmodel_new = model_new.to(device)\ninput_tensor = input_tensor.to(device)\n\nwith torch.no_grad():\n    outputs = model_new(input_tensor)\n    probabilities = torch.softmax(outputs, dim=1)\n    prediction = torch.argmax(probabilities, dim=1)\n\nprint(f'Model prediction: Class {prediction.item()}')\nprint(f'Confidence: {probabilities[0, prediction.item()].item():.4f}')\nprint(f'\\nAll class probabilities:')\nfor i, prob in enumerate(probabilities[0]):\n    print(f'  Class {i}: {prob.item():.4f}')"]]

## 7.4 Best Practices

**Always save:**
- Model weights/state dict
- Hyperparameters and training settings
- Training history and metrics
- Date and architecture info

**Example: Complete checkpoint**

In [ ]:
["import json\nfrom datetime import datetime\n\n# Create complete checkpoint\ncomplete_checkpoint = {\n    'timestamp': datetime.now().isoformat(),\n    'model': {\n        'architecture': 'ResNet-18',\n        'num_classes': 10,\n        'state_dict': model.state_dict(),\n    },\n    'training': {\n        'epochs': 10,\n        'batch_size': 32,\n        'learning_rate': 0.001,\n        'optimizer': 'Adam',\n    },\n    'performance': {\n        'train_accuracy': 0.95,\n        'val_accuracy': 0.92,\n        'test_accuracy': 0.90,\n    },\n    'data': {\n        'dataset': 'CIFAR-10',\n        'train_samples': 50000,\n        'test_samples': 10000,\n    }\n}\n\n# Save\ntorch.save(complete_checkpoint, 'complete_checkpoint.pth')\nprint('Complete checkpoint saved!')\nprint('\\nCheckpoint contents:')\nprint(f'  - Model: {complete_checkpoint[\"model\"][\"architecture\"]}')\nprint(f'  - Classes: {complete_checkpoint[\"model\"][\"num_classes\"]}')\nprint(f'  - Test Accuracy: {complete_checkpoint[\"performance\"][\"test_accuracy\"]:.4f}')\nprint(f'  - Saved at: {complete_checkpoint[\"timestamp\"]}')"]]